# Pipeline de Datos - History-ARG

Este notebook descarga, procesa y prepara los datos de las fuentes SIDE y RUVTE para el sistema RAG.

**Fuentes:**
- **SIDE:** 26 documentos desclasificados (987 páginas), OCR comunitario de [side.com.ar](https://side.com.ar/)
- **RUVTE:** 9,415 víctimas del terrorismo de estado (CSVs oficiales)

**Output:** `chunks.jsonl` con ~10,000+ documentos listos para embedding.

## 1. Setup

In [ ]:
# En Colab, descomentar las siguientes líneas:
# !pip install -q langchain pandas tqdm requests
# !git clone https://github.com/TU_USUARIO/History-ARG.git
# %cd History-ARG

import sys
from pathlib import Path

# Asegurar que src/ esté en el path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "raw"
OUTPUT_DIR = ROOT / "data" / "processed"

print(f"Root: {ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

## 2. Descargar datos

### 2.1 SIDE - OCR comunitario (987 páginas)

In [ ]:
from src.data_acquisition import download_side_ocr, download_side_metadata

side_ocr_dir = DATA_DIR / "side" / "ocr"

# Descargar los 987 JSONs de OCR (usa cache si ya existen)
ocr_pages = download_side_ocr(str(side_ocr_dir), max_workers=10)
print(f"\nTotal páginas OCR descargadas: {len(ocr_pages)}")
print(f"Primera página: {ocr_pages[0]['doc_id']} - confianza: {ocr_pages[0]['confidence']}")
print(f"Texto preview: {ocr_pages[0]['text'][:200]}...")

In [ ]:
# Descargar metadata de documentos y carpetas
side_meta_dir = DATA_DIR / "side"
documents_meta, carpetas_meta = download_side_metadata(str(side_meta_dir))

print(f"Documentos: {len(documents_meta)}")
print(f"Carpetas: {len(carpetas_meta)}")
print(f"\nEjemplo de documento:")
doc = documents_meta[0]
print(f"  ID: {doc['id']}")
print(f"  Título: {doc['title']}")
print(f"  Fecha: {doc['date']}")
print(f"  Tipo: {doc['type']}")
print(f"  Páginas: {doc['page_count']}")
print(f"  Tags: {doc.get('tags', [])}")

### 2.2 RUVTE - Registro de Víctimas

In [ ]:
from src.data_acquisition import download_ruvte

ruvte_dir = DATA_DIR / "ruvte"
df_con, df_sin = download_ruvte(str(ruvte_dir))

print(f"\nRUVTE con denuncia: {len(df_con)} registros")
print(f"RUVTE sin denuncia: {len(df_sin)} registros")
print(f"Total: {len(df_con) + len(df_sin)} víctimas")
print(f"\nColumnas: {list(df_con.columns)}")
print(f"\nTipificaciones:")
print(df_con['tipificacion_ruvte'].value_counts())

## 3. Procesar datos

### 3.1 SIDE - Agrupar páginas por documento y chunkear

In [ ]:
from src.text_processing import build_side_documents, chunk_side_documents

# Agrupar páginas OCR por documento y enriquecer con metadata
side_docs = build_side_documents(ocr_pages, documents_meta)

print(f"Documentos SIDE agrupados: {len(side_docs)}")
for doc in side_docs[:3]:
    print(f"  {doc['doc_id']}: {doc['title'][:60]}... ({doc['page_count']} págs, {len(doc['text'])} chars)")

In [ ]:
# Chunkear los documentos SIDE
side_chunks = chunk_side_documents(side_docs, chunk_size=1000, chunk_overlap=200)

print(f"Chunks SIDE generados: {len(side_chunks)}")
print(f"\nEjemplo de chunk:")
print(f"  Texto: {side_chunks[0]['text'][:150]}...")
print(f"  Metadata: {side_chunks[0]['metadata']}")

### 3.2 RUVTE - Convertir registros a texto natural

In [ ]:
from src.text_processing import process_ruvte, ruvte_record_to_text

# Preview de cómo se ve un registro convertido a texto
print("Ejemplo de registro RUVTE como texto natural:")
print("-" * 60)
print(ruvte_record_to_text(df_con.iloc[0]))
print("-" * 60)

# Procesar todos los registros
ruvte_chunks = process_ruvte(df_con, df_sin)
print(f"\nChunks RUVTE generados: {len(ruvte_chunks)}")

## 4. Combinar y guardar

In [ ]:
from src.text_processing import save_chunks

# Combinar todos los chunks
all_chunks = side_chunks + ruvte_chunks

print(f"Total chunks:")
print(f"  SIDE: {len(side_chunks)}")
print(f"  RUVTE: {len(ruvte_chunks)}")
print(f"  TOTAL: {len(all_chunks)}")

# Guardar como JSONL
output_path = OUTPUT_DIR / "chunks.jsonl"
save_chunks(all_chunks, str(output_path))

## 5. Estadísticas del corpus

In [ ]:
import statistics

# Estadísticas de longitud de chunks
side_lens = [len(c['text']) for c in side_chunks]
ruvte_lens = [len(c['text']) for c in ruvte_chunks]

print("Estadísticas de longitud de texto (caracteres):")
print(f"\nSIDE ({len(side_chunks)} chunks):")
print(f"  Min: {min(side_lens)}, Max: {max(side_lens)}")
print(f"  Media: {statistics.mean(side_lens):.0f}, Mediana: {statistics.median(side_lens):.0f}")

print(f"\nRUVTE ({len(ruvte_chunks)} chunks):")
print(f"  Min: {min(ruvte_lens)}, Max: {max(ruvte_lens)}")
print(f"  Media: {statistics.mean(ruvte_lens):.0f}, Mediana: {statistics.median(ruvte_lens):.0f}")

# Estimación para Cloudflare Vectorize
total_docs = len(all_chunks)
dims = 384  # bge-small
total_dims = total_docs * dims
free_limit = 5_000_000
print(f"\nEstimación Vectorize (bge-small, {dims}d):")
print(f"  Total vectores: {total_docs:,}")
print(f"  Total dimensiones: {total_dims:,}")
print(f"  Free tier (5M): {'OK' if total_dims <= free_limit else 'EXCEDE'} ({total_dims/free_limit*100:.1f}%)")

## Siguiente paso

El archivo `chunks.jsonl` está listo. El siguiente paso es la **Fase 3**: generar embeddings y subirlos a Cloudflare Vectorize + D1.